In [1]:
import sys; sys.path.insert(0, ".")
import pandas as pd
from src.data.db import get_connection
from src.processing.xg_processing import load_shots

conn = get_connection()
print(pd.read_sql("SELECT DISTINCT event_type FROM shot_events", conn))

df = load_shots(conn)
print(len(df))   # expect 566,545

     event_type
0  shot-on-goal
1  blocked-shot
2   missed-shot
3          goal
566545


In [2]:
from src.processing.xg_processing import load_shots, normalize_coordinates

df = load_shots(conn)
print((df.x_coord > 25).mean())    # before: about 0.50
df = normalize_coordinates(df)
print((df.x_coord > 25).mean())    # after: about 0.99

low = df[df.x_coord <= 25]

# 1. What zone does the API say these shots came from?
print(low.zone_code.value_counts())

# 2. Offensive-zone shots that ended up past the blue line, should be ~1.0
o = df[df.zone_code == "O"]
print((o.x_coord > 25).mean())

# 3. Where do the low shots sit?
print(pd.cut(low.x_coord, [-100, -25, 0, 25]).value_counts())
print(df.zone_code.value_counts(normalize=True))

o = df[df.zone_code == "O"]
n = df[df.zone_code == "N"]
print(o.x_coord.quantile(0.01))   # the shallowest offensive-zone shots
print(n.x_coord.max())            # the deepest neutral-zone shots

0.48421572867115587
0.9575550044568393
zone_code
N    14482
D     9069
O      496
Name: count, dtype: int64
0.9990864939231586
x_coord
(0, 25]        10745
(-100, -25]     9106
(-25, 0]        4196
Name: count, dtype: int64
zone_code
O    0.958376
N    0.025562
D    0.016062
Name: proportion, dtype: float64
28.0
24.0


In [3]:
import sys; sys.path.insert(0, "..")
import pandas as pd
from src.data.db import get_connection

conn = get_connection()
print(pd.read_sql("""
    SELECT season,
           COUNT(*)            AS shots,
           ROUND(SUM(xg), 1)   AS total_xg,
           ROUND(MIN(xg), 4)   AS min_xg,
           ROUND(MAX(xg), 3)   AS max_xg
    FROM shot_xg
    GROUP BY season
""", conn))

     season   shots  total_xg  min_xg  max_xg
0  20212022  112561    7922.2  0.0004   0.884
1  20222023  113351    8051.3  0.0003   0.860
2  20232024  114059    7438.8  0.0002   0.869
3  20242025  111304    7150.7  0.0003   0.882
4  20252026  111099    7759.4  0.0004   0.867


In [4]:
"""
team_strength.py
=================
Rolling team offense and defense estimates from shot-level xG.

Home: src/processing/team_strength.py
"""
import sys; sys.path.insert(0, "..")
import numpy as np
import pandas as pd
from src.data.db import get_connection

conn = get_connection()

import sqlite3

import numpy as np
import pandas as pd



rows = pd.read_sql("""
    SELECT game_id, game_date, season, 
        home_team_id as team_id, home_team_abbrev AS team,
        away_team_abbrev AS opponent, 1 AS is_home
    FROM schedule
    WHERE game_id IN (SELECT DISTINCT game_id FROM shot_xg)

    UNION ALL

    SELECT game_id, game_date, season, 
            away_team_id, away_team_abbrev,
            home_team_abbrev, 0
    FROM schedule
    WHERE game_id IN (SELECT DISTINCT game_id FROM shot_xg)

    ORDER BY game_date, game_id, is_home DESC
""", conn)

print(len(rows))                        # expect 13,120
print(rows.groupby("season").size())    # expect 2,624 per season
print(rows.is_home.mean())              # expect 0.5
print(rows.head(4))
#2 Sum shot_xg.xg per (game_id,tean_id) -> xgf




13120
season
20212022    2624
20222023    2624
20232024    2624
20242025    2624
20252026    2624
dtype: int64
0.5
      game_id   game_date    season  team_id team opponent  is_home
0  2021020001  2021-10-12  20212022       14  TBL      PIT        1
1  2021020001  2021-10-12  20212022        5  PIT      TBL        0
2  2021020002  2021-10-12  20212022       54  VGK      SEA        1
3  2021020002  2021-10-12  20212022       55  SEA      VGK        0


In [5]:
from src.processing.team_strength import team_game_xg
tg = team_game_xg(conn)
print(len(tg))                                        # expect 13,120
print(tg.groupby("season").xgf.sum().round(1))        # should match the score_xg totals
print(abs(tg.xgf.sum() - tg.xga.sum()) < 1e-6)        # True: league xGF = league xGA
print(tg.minutes.describe().round(1))                 # min 60, max 65
print((tg.gf + tg.ga).groupby(tg.season).sum() / 2)   # goals per season, ~7,700-8,000

13120
season
20212022    7922.2
20222023    8051.3
20232024    7438.8
20242025    7150.7
20252026    7759.4
Name: xgf, dtype: float64
True
count    13120.0
mean        60.7
std          1.6
min         60.0
25%         60.0
50%         60.0
75%         60.0
max         65.0
Name: minutes, dtype: float64
season
20212022    8150.0
20222023    8248.0
20232024    8086.0
20242025    7901.0
20252026    8086.0
dtype: float64


In [6]:
from src.processing.team_strength import team_game_xg, rolling_strength

tg = team_game_xg(conn)
rs = rolling_strength(tg)

print(rs.xgf60.isna().sum())                          # first game per team: ~33
print(rs.games_in_window.value_counts().sort_index().tail(3))   # mostly 25
print(rs[["xgf60", "xga60"]].describe().round(2))     # mean ~2.5-3.0, both similar

# end of 2025-26: the best and worst offenses
last = rs[rs.season == 20252026].groupby("team").tail(1)
print(last.sort_values("xgf60", ascending=False)[["team", "xgf60", "xga60", "xgf60_var"]].head(5).round(3))

season = tg[tg.season == 20252026].groupby("team")[["xgf", "xga", "minutes"]].sum()
season["xgf60"] = season.xgf / season.minutes * 60
season["xga60"] = season.xga / season.minutes * 60
print(season.sort_values("xgf60", ascending=False).round(2).head(10))

33
games_in_window
23       33
24       33
25    12295
Name: count, dtype: int64
          xgf60     xga60
count  13087.00  13087.00
mean       2.89      2.89
std        0.32      0.31
min        1.93      1.35
25%        2.67      2.67
50%        2.86      2.87
75%        3.09      3.08
max        5.44      5.93
     team  xgf60  xga60  xgf60_var
409   ANA  3.550  3.212      0.029
3525  COL  3.402  2.718      0.043
8445  OTT  3.386  2.555      0.025
1885  CAR  3.327  2.589      0.031
9265  PIT  3.325  3.134      0.033
         xgf     xga  minutes  xgf60  xga60
team                                       
COL   280.20  219.78  4984.80   3.37   2.65
CAR   277.19  219.88  4996.90   3.33   2.64
EDM   269.59  249.30  4970.48   3.25   3.01
ANA   269.07  254.14  5010.52   3.22   3.04
OTT   264.09  207.11  4966.95   3.19   2.50
PIT   265.02  256.08  5003.53   3.18   3.07
VGK   262.69  207.85  5015.67   3.14   2.49
TBL   255.18  224.92  4970.97   3.08   2.71
NJD   250.24  240.28  4974.13   3.0

In [7]:
from src.processing.team_strength import team_game_xg, strength_as_of

tg = team_game_xg(conn)

# early season: about 10 games into 2025-26
early = strength_as_of(tg, "2025-10-25")
print(early.sort_values("xgf60", ascending=False).round(3).head(10))
print(early.games_in_window.value_counts())   # all 25, thanks to the 2024-25 carry-over
print(len(early))                             # 32 teams

   team  games_in_window   last_game  xgf60  xga60  xgf60_var  xga60_var
4   CAR               25  2025-10-23  3.170  2.718      0.023      0.026
8   COL               25  2025-10-23  3.052  2.342      0.022      0.012
12  FLA               25  2025-10-23  3.034  2.400      0.012      0.016
18  NYI               25  2025-10-23  3.020  3.016      0.013      0.018
11  EDM               25  2025-10-23  2.979  2.563      0.053      0.030
0   ANA               25  2025-10-23  2.975  3.076      0.037      0.025
26  TBL               25  2025-10-23  2.945  2.665      0.028      0.030
32  WSH               25  2025-10-24  2.943  2.620      0.026      0.025
5   CBJ               25  2025-10-24  2.877  3.048      0.027      0.018
28  UTA               25  2025-10-23  2.853  2.692      0.027      0.024
games_in_window
25    33
Name: count, dtype: int64
33


In [10]:
from src.processing.goalie import goalie_game_xg

gg = goalie_game_xg(conn)
print(len(gg))                                        # roughly 13,500-14,000
print(gg.groupby("season").agg(xga=("xga", "sum"), ga=("ga", "sum"), gsax=("gsax", "sum")).round(1))
print(gg.goalie_name.isna().mean())                   # share of rows with no name
print(gg.groupby("goalie_name").gsax.sum().sort_values(ascending=False).head(10).round(1))

print(gg.goalie_name.isna().mean())    # expect 0.0
top = gg.groupby(["goalie_id", "goalie_name"]).gsax.sum().sort_values(ascending=False)
print(top.head(10).round(1))

13919
             xga    ga   gsax
season                       
20212022  7922.2  7674  248.2
20222023  8051.3  7810  241.3
20232024  7438.8  7639 -200.2
20242025  7150.7  7377 -226.3
20252026  7759.4  7578  181.4
0.15453696386234644
goalie_name
Igor Shesterkin       123.6
Connor Hellebuyck     107.5
Ilya Sorokin          104.0
Andrei Vasilevskiy     92.3
Linus Ullmark          58.7
Juuse Saros            54.3
Logan Thompson         46.3
Anthony Stolarz        45.2
Jeremy Swayman         43.7
Frederik Andersen      40.6
Name: gsax, dtype: float64
0.15453696386234644
goalie_id  goalie_name       
8478048    Igor Shesterkin       123.6
8476945    Connor Hellebuyck     107.5
8478009    Ilya Sorokin          104.0
8476883    Andrei Vasilevskiy     92.3
8476999    Linus Ullmark          58.7
8477424    Juuse Saros            54.3
8480313    Logan Thompson         46.3
8476932    Anthony Stolarz        45.2
8480280    Jeremy Swayman         43.7
8475883    Frederik Andersen      40.6
Name: